In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load all Olist tables
def load_olist_data(data_dir):
    """Load all Olist CSV files into a dictionary of DataFrames."""

    tables = {
        'orders': 'olist_orders_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'customers': 'olist_customers_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'category_translation': 'product_category_name_translation.csv'
    }

    # Date columns to parse
    date_cols = {
        'orders': ['order_purchase_timestamp', 'order_approved_at',
                   'order_delivered_carrier_date', 'order_delivered_customer_date',
                   'order_estimated_delivery_date'],
        'order_items': ['shipping_limit_date'],
        'reviews': ['review_creation_date', 'review_answer_timestamp']
    }

    data = {}
    for name, filename in tables.items():
        filepath = data_dir + "/" + filename
        parse_dates = date_cols.get(name, None)
        data[name] = pd.read_csv(filepath, parse_dates=parse_dates)
        print(f"Loaded {name}: {data[name].shape[0]:,} rows × {data[name].shape[1]} cols")

    return data

# Load data
DATA_DIR = './raw/Olist'
olist = load_olist_data(DATA_DIR)

Loaded orders: 99,441 rows × 8 cols
Loaded order_items: 112,650 rows × 7 cols
Loaded customers: 99,441 rows × 5 cols
Loaded products: 32,951 rows × 9 cols
Loaded sellers: 3,095 rows × 4 cols
Loaded payments: 103,886 rows × 5 cols
Loaded reviews: 99,224 rows × 7 cols
Loaded geolocation: 1,000,163 rows × 5 cols
Loaded category_translation: 71 rows × 2 cols


In [3]:
def dupes_and_na(dataset):
    na = dataset.isna().sum()
    idx = []
    val = []
    for col in dataset.columns:
        idx.append(col)
        val.append(dataset[col].duplicated().sum())
    dupe = pd.Series(val,index=idx)
    data = {'NA Values': na, 'Duplicated Values': dupe}
    return pd.DataFrame.from_dict(data)

for name, df in olist.items():
    print(f'{name.upper()}\n{dupes_and_na(df)}')
    print(f'duplicated rows: {df.duplicated().sum()}\n')

ORDERS
                               NA Values  Duplicated Values
order_id                               0                  0
customer_id                            0                  0
order_status                           0              99433
order_purchase_timestamp               0                566
order_approved_at                    160               8707
order_delivered_carrier_date        1783              18422
order_delivered_customer_date       2965               3776
order_estimated_delivery_date          0              98982
duplicated rows: 0

ORDER_ITEMS
                     NA Values  Duplicated Values
order_id                     0              13984
order_item_id                0             112629
product_id                   0              79699
seller_id                    0             109555
shipping_limit_date          0              19332
price                        0             106682
freight_value                0             105651
duplicated rows: 0

C

# Key datasets


In [4]:
def show_quality(name:str, dataset: pd.DataFrame):
    """Describes dataset's shape and shows NA and duplicates in each column"""
    
    label = name.upper()+' TABLE'
    print(f'{label}\n{'='*40}')
    print(f'Shape: {dataset.shape}  Length: {len(dataset)}')
    print(f'{dataset.dtypes}')
    print(f'\nNA VALUES\n{'-'*40}')
    print(f'{dataset.isna().sum()}\n')
    print(f'VALUES WITH DUPLICATES\n{'-'*40}')
    for col in dataset.columns:
        print(f'{col:35} {(dataset[col].value_counts()>1).sum()}')

## Orders Table

In [5]:
show_quality('orders', olist['orders'])
print(f"\nOrder status distribution:\n{olist['orders']['order_status'].value_counts()}")
olist['orders'].head(3)

ORDERS TABLE
Shape: (99441, 8)  Length: 99441
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

NA VALUES
----------------------------------------
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

VALUES WITH DUPLICATES
----------------------------------------
order_id                            0
customer_id                         0
order_status                        8
order_pu

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


## Order Items Table

In [6]:
show_quality('order items',olist['order_items'])
print(f"\nOrder price distribution:\n{olist['order_items']['price'].describe()}")
print(f"\nFreight value distribution:\n{olist['order_items']['freight_value'].describe()}")
olist['order_items'].head(3)

ORDER ITEMS TABLE
Shape: (112650, 7)  Length: 112650
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

NA VALUES
----------------------------------------
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

VALUES WITH DUPLICATES
----------------------------------------
order_id                            9803
order_item_id                       20
product_id                          14834
seller_id                           2586
shipping_limit_date                 13482
price                               3626
freight_value                       4924

Order price distribution:
count    112650.000000
mean        120.653739
st

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


## Products Table

In [7]:
products = olist['products'].merge(
    olist['category_translation'], 
    on = 'product_category_name', 
    how = 'left')

show_quality('products', products)
print(f"\nProduct category distribution:\n{products['product_category_name_english'].value_counts()}")
products.head(3)

PRODUCTS TABLE
Shape: (32951, 10)  Length: 32951
product_id                           str
product_category_name                str
product_name_lenght              float64
product_description_lenght       float64
product_photos_qty               float64
product_weight_g                 float64
product_length_cm                float64
product_height_cm                float64
product_width_cm                 float64
product_category_name_english        str
dtype: object

NA VALUES
----------------------------------------
product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    623
dtype: int64

VALUES WITH DUPLICATES
----------------------------------------
product_id                 

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure


## Payments Table

In [8]:
show_quality('payments',olist['payments'])
print(f"\nPayment type distribution:\n{olist['payments']['payment_type'].value_counts()}")
print(f"\nInstallment distribution:\n{olist['payments']['payment_installments'].value_counts()}")
olist['payments'].head(3)

PAYMENTS TABLE
Shape: (103886, 5)  Length: 103886
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

NA VALUES
----------------------------------------
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

VALUES WITH DUPLICATES
----------------------------------------
order_id                            2961
payment_sequential                  26
payment_type                        5
payment_installments                22
payment_value                       15978

Payment type distribution:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Installment distribution:
payment_installments
1     52546
2     12413
3     10461
4      7098
10     5328
5      5239
8      4268
6      3920
7

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


## Reviews Table

In [9]:
show_quality('reviews',olist['reviews'])
print(f"\nReview score distribution:\n{olist['reviews']['review_score'].value_counts().sort_index(ascending=False)}")
print(f"\nReviews with title (%): {olist['reviews']['review_comment_title'].count()/olist['reviews']['review_id'].count():.1%}")
print(f"\nReviews with comments (%): {olist['reviews']['review_comment_message'].notna().mean():.1%}")
olist['reviews'].head(3)

REVIEWS TABLE
Shape: (99224, 7)  Length: 99224
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

NA VALUES
----------------------------------------
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

VALUES WITH DUPLICATES
----------------------------------------
review_id                           789
order_id                            547
review_score                        5
review_comment_title                764
review_comment_message              1182
review_creation_date                607
review_answer_timestamp             945

Revi

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24


# Data Quality

## Reviews

In [10]:
def check_id_dupes(reviewtable, *id_name):
    """Displays summary of unique and duplicate ID rows"""
    result = []
    for id in id_name:
        print(f'\n{id}\n{'-'*50}')
        print(f'Total number of rows = Number of unique ids? {len(reviewtable) == len(reviewtable['order_id'].unique())}')
        dupe_id = reviewtable['order_id'].duplicated().sum()
        unique_id = len(reviewtable['order_id'].unique())
        print(f'Total number of rows: {len(reviewtable)}\nNumber of duplicated {id}: {dupe_id}\nTotal number of unique {id}: {unique_id}')
        result.append(len(reviewtable[id].unique()))

    return tuple(result)

def row_difference(before, after):
    """Helper function to display row loss"""
    difference = before - after
    print(f'{before} - {after} = {difference}')
    return difference

In [41]:
before_unique_oid, before_unique_rid = check_id_dupes(olist['reviews'], 'order_id', 'review_id')


order_id
--------------------------------------------------
Total number of rows = Number of unique ids? False
Total number of rows: 99224
Number of duplicated order_id: 551
Total number of unique order_id: 98673

review_id
--------------------------------------------------
Total number of rows = Number of unique ids? False
Total number of rows: 99224
Number of duplicated review_id: 551
Total number of unique review_id: 98673


### Issues with Reviews dataset
1. `.duplicated().sum()` returns 0 despite there being duplicate `order_id` and duplicate `review_id`
* (a) Reviews with the same `review_id` (true duplicates, just generated under different ids)
    * Solution: Drop duplicated rows
    * Justification: Each review is tied to an `order_id`, therefore there should only be one review one order.
* (b) Reviews for the same `order_id` at different creation and answer timestamps. These could contain same or different review messages.
    * Solution: Take the latest review as the final review for the order.
    * Justification: The marketing team wants to come up with higher customer satisfaction. The most updated review will reveal the latest customer satisfaction results, thus there is no need to consider past reviews in the analysis.
* Below code block resolves this issue

In [42]:
df = olist['reviews'].copy()

### 1 (a) Rows with duplicated Review IDs

In [43]:
before_drop = df.shape[0]
df.drop_duplicates(subset=['review_id'], inplace=True)

after_drop = df.shape[0]

print(f'Remaining duplicate review_id rows: {df['review_id'].duplicated().sum()}')
print(f'Original length of Review table - Length of Review table after dropping duplicate review_id:\n  {before_drop} - {after_drop} = {before_drop-after_drop}')
print(f'Remaining duplicate order_id rows: {df['order_id'].duplicated().sum()}')
print(f'Unique order_id: {len(df['order_id'].unique())}')

Remaining duplicate review_id rows: 0
Original length of Review table - Length of Review table after dropping duplicate review_id:
  99224 - 98410 = 814
Remaining duplicate order_id rows: 243
Unique order_id: 98167


### 1 (b) Rows with duplicated Order IDs

In [44]:
df = df.join((df['order_id'].value_counts() > 1).rename('id_dupe'), on = 'order_id')
df.head()
df_unique = df.loc[df['id_dupe']==False]
df_unique[df_unique['id_dupe']==False].count() # 97924 unique ids

df_dupes = df.loc[df['id_dupe']!=False]

df_dupes = df_dupes.sort_values(by='review_answer_timestamp', ascending=False).drop_duplicates(subset='order_id').sort_index()

df_dupes.head()

df_merged = pd.concat([df_unique, df_dupes]).sort_index()
df_merged.info()

<class 'pandas.DataFrame'>
Index: 98167 entries, 0 to 99223
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                98167 non-null  str           
 1   order_id                 98167 non-null  str           
 2   review_score             98167 non-null  int64         
 3   review_comment_title     11513 non-null  str           
 4   review_comment_message   40575 non-null  str           
 5   review_creation_date     98167 non-null  datetime64[us]
 6   review_answer_timestamp  98167 non-null  datetime64[us]
 7   id_dupe                  98167 non-null  bool          
dtypes: bool(1), datetime64[us](2), int64(1), str(4)
memory usage: 6.1 MB


In [45]:
after_unique_oid, after_unique_rid = check_id_dupes(df_merged, 'order_id', 'review_id')


order_id
--------------------------------------------------
Total number of rows = Number of unique ids? True
Total number of rows: 98167
Number of duplicated order_id: 0
Total number of unique order_id: 98167

review_id
--------------------------------------------------
Total number of rows = Number of unique ids? True
Total number of rows: 98167
Number of duplicated review_id: 0
Total number of unique review_id: 98167


### Data Loss

In [46]:
print('Difference in number of unique order_id:')
oid_diff = row_difference(before_unique_oid, after_unique_oid)
print(f'Percentage loss: {oid_diff/before_unique_oid:.1%}\n')
print('Difference in number of unique review_id:')
rid_diff = row_difference(before_unique_rid, after_unique_rid)
print(f'Percentage loss: {rid_diff/before_unique_rid:.1%}')

Difference in number of unique order_id:
98673 - 98167 = 506
Percentage loss: 0.5%

Difference in number of unique review_id:
98410 - 98167 = 243
Percentage loss: 0.2%


### Adding Score Class
Classifiying review score as 'Low' if less than 4, otherwise 'High'

In [47]:
df_merged["score_class"] = np.where(df_merged["review_score"]<4,"Low","High")

reviews = df_merged.drop(columns=['id_dupe','review_creation_date','review_answer_timestamp'])
reviews.info()
print(f'Shape of cleaned Reviews table: {reviews.shape}')

<class 'pandas.DataFrame'>
Index: 98167 entries, 0 to 99223
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   review_id               98167 non-null  str  
 1   order_id                98167 non-null  str  
 2   review_score            98167 non-null  int64
 3   review_comment_title    11513 non-null  str  
 4   review_comment_message  40575 non-null  str  
 5   score_class             98167 non-null  str  
dtypes: int64(1), str(5)
memory usage: 5.2 MB
Shape of cleaned Reviews table: (98167, 6)


## Payments

In [18]:
# Investigating Payments
payments = olist['payments']

print(payments['payment_type'].value_counts())
before_unique_oid = check_id_dupes(payments,'order_id')

payments.loc[payments['payment_sequential']>1,].head()
payments.loc[payments['order_id']=='ea9184ad433a404df1d72fa0a8764232',] #implies 1 order_id : many payments

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

order_id
--------------------------------------------------
Total number of rows = Number of unique ids? False
Total number of rows: 103886
Number of duplicated order_id: 4446
Total number of unique order_id: 99440


,order_id,payment_sequential,payment_type,payment_installments,payment_value
121,ea9184ad433a404df1d72fa0a8764232,4,voucher,1,22.49
40168,ea9184ad433a404df1d72fa0a8764232,1,credit_card,1,17.78
78613,ea9184ad433a404df1d72fa0a8764232,3,voucher,1,22.49
94151,ea9184ad433a404df1d72fa0a8764232,2,voucher,1,22.49
99425,ea9184ad433a404df1d72fa0a8764232,5,voucher,1,22.47


### Issues with Payments dataset
2. Key dataset is sorted by line item (from `order_items`). However, a single order can be tagged to multiple payments. This disrupts the unit of analysis.
* (a) `payment_sequential` refers to the sequence of the payment method the customer used. This does not refer to the number of payment methods the customer has employed for the specific order, rather the nominal number of the payment method used.
    * Solution: Retrieve the maximum number from a given `order_id`
    * Justification: It is part of the EDA to figure out if this is actually a useful point of data or not. Since the individual numbers themselves do not mean much, the highest sequence recorded should be used in this case.
* (b) `payment_installments` refers to the number of instalments chosen by the customer. Multiple installment payments will take up multiple rows.
    * Solution: Group payments by `order_id`, then retrieve the sum total of instalments
    * Justification: Aggregating the instalments according to `order_id` to find the sum total in order to tie 1 order to number of instalments made
* (c) `payment_type` refers to the method of payment the customer has chosen. A customer may use multiple payment methods for a single order.
    * Solution: Grouping payments by `order_id`, and only store unique methods. Then, if more than 1 method is used, rename to 'Multiple'.
    * Justification: Proportion of orders with multiple payment methods is small enough to justify grouping them as 1 category
* (d) `payment_value` refers to the amount paid by the customer during a transaction. It is the same value as the price and freight value from the `order_items` dataset.
    * Solution: Drop the column
    * Justification: The resulting aggregation is a repeat of an already existing column.


In [19]:
df = olist['payments'].copy()
df.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


### 2(a) Retrieve only unique Order IDs

In [20]:
df_uniques = df.sort_values(by='payment_sequential', ascending=False).drop_duplicates(subset='order_id').sort_index()
df_uniques.describe() == df.describe()

,payment_sequential,payment_installments,payment_value
count,False,False,False
mean,False,False,False
std,False,False,False
min,True,True,True
25%,True,True,False
50%,True,False,False
75%,True,True,False
max,True,True,True


### 2 (b) Aggregate total number of payment sequentials

In [21]:
df_uniques['payment_installments_total'] = df.groupby(['order_id'])['payment_installments'].transform(lambda x: sum(x))
df_uniques.describe()

,payment_sequential,payment_installments,payment_value,payment_installments_total
count,99440.000000,99440.000000,99440.000000,99440.000000
mean,1.045515,2.901026,158.267094,2.980923
std,0.382177,2.702330,218.962413,2.741810
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,60.220000,1.000000
50%,1.000000,2.000000,103.270000,2.000000
75%,1.000000,4.000000,175.080000,4.000000
max,29.000000,24.000000,13664.080000,29.000000


### 2(c) Aggregate payment types

In [22]:
df_uniques['payment_type_all'] = df.groupby(['order_id'])['payment_type'].transform(lambda x: ', '.join(x)).transform(lambda x: 'multiple' if ',' in x else x)
# people who use the same payment method but for different installments are counted as 'multiple'
print(df_uniques['payment_type_all'].value_counts())
df_uniques.describe()

payment_type_all
credit_card    73972
boleto         19784
multiple        2961
debit_card      1526
voucher         1194
not_defined        3
Name: count, dtype: int64


,payment_sequential,payment_installments,payment_value,payment_installments_total
count,99440.000000,99440.000000,99440.000000,99440.000000
mean,1.045515,2.901026,158.267094,2.980923
std,0.382177,2.702330,218.962413,2.741810
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,60.220000,1.000000
50%,1.000000,2.000000,103.270000,2.000000
75%,1.000000,4.000000,175.080000,4.000000
max,29.000000,24.000000,13664.080000,29.000000


### 2(d) Drop payment value

In [23]:
payments = df_uniques.drop(columns=['payment_installments', 'payment_type', 'payment_value'])
print(f'Shape after cleaning: {payments.shape}')
payments.head(5)

Shape after cleaning: (99440, 4)


,order_id,payment_sequential,payment_installments_total,payment_type_all
0,b81ef226f3fe1789b1e8b2acac839d17,1,8,credit_card
1,a9810da82917af2d9aefd1278f1dcfa0,1,1,credit_card
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,1,credit_card
3,ba78997921bbcdc1373bb41e913ab953,1,8,credit_card
4,42fdf880ba16b47b59251dd489d4441a,1,2,credit_card


In [24]:
after_unique_oid = check_id_dupes(payments, 'order_id')


order_id
--------------------------------------------------
Total number of rows = Number of unique ids? True
Total number of rows: 99440
Number of duplicated order_id: 0
Total number of unique order_id: 99440


### Data Loss

In [25]:
print('Difference in number of unique order_id:')
oid_diff = row_difference(before_unique_oid[0], after_unique_oid[0])
print(f'Percentage loss: {oid_diff/before_unique_oid[0]:.1%}\n')

Difference in number of unique order_id:
99440 - 99440 = 0
Percentage loss: 0.0%



# Merging datasets

In [48]:
df = olist['order_items'].copy()
print(f'{'Initialise':12}: Order Items {df.shape}')

df = df.join(products.set_index("product_id"), on="product_id")
print(f'{'Merged with':12}: Products {df.shape}\n    Shape = {payments.shape}')

df = df.join(olist['orders'].set_index('order_id'), on='order_id')
print(f'{'Merged with':12}: Orders {df.shape}\n    Shape = {payments.shape}')

df = df.join(reviews.set_index('order_id'), on='order_id')
print(f'{'Merged with':12}: Reviews {df.shape}\n    Shape = {reviews.shape}')

df = df.join(payments.set_index('order_id'), on='order_id')
print(f'{'Merged with':12}: Payments {df.shape}\n    Shape = {payments.shape}')

df = df.join(olist['sellers'].set_index('seller_id'), on='seller_id')
print(f'{'Merged with':12}: Sellers {df.shape}\n    Shape = {olist['sellers'].shape}')

df = df.join(olist['customers'].set_index('customer_id'), on='customer_id')
print(f'{'Merged with':12}: Customers {df.shape}\n    Shape = {olist['customers'].shape}')

Initialise  : Order Items (112650, 7)
Merged with : Products (112650, 16)
    Shape = (99440, 4)
Merged with : Orders (112650, 23)
    Shape = (99440, 4)
Merged with : Reviews (112650, 28)
    Shape = (98167, 6)
Merged with : Payments (112650, 31)
    Shape = (99440, 4)
Merged with : Sellers (112650, 34)
    Shape = (3095, 4)
Merged with : Customers (112650, 38)
    Shape = (99441, 5)


## Feature engineer datetime columns

In [49]:
datetime_col = [
    'shipping_limit_date', 
    'order_purchase_timestamp', 
    'order_approved_at',
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
    ]

# NA approval dates replaced with purchase timestamp
df['order_approved_at'] = np.where(df['order_approved_at'].isna() == True, df['order_purchase_timestamp'], df['order_approved_at'])

def duration_convertor(date1, date2):
    """Converts datetime format into integers"""
    return (df[date1] - df[date2]).dt.days

df['approval_duration'] = duration_convertor('order_approved_at', 'order_purchase_timestamp')
df['customer_eta_days'] = duration_convertor('order_estimated_delivery_date', 'order_approved_at')

df['carrier_offset'] = duration_convertor('shipping_limit_date', 'order_delivered_carrier_date')
df['customer_offset'] = duration_convertor('order_estimated_delivery_date', 'order_delivered_customer_date')

def convert_dt_offsets(df, offset_col):
    """Converts datetime duration into categories of late, early, on-time and not delivered"""
    ctype = offset_col[:len(offset_col)-len('_offset')]
    cat = 'delivered_'+ctype
    num = 'delivered_'+ctype+'_days'
    
    print(f'{offset_col}\n{df[offset_col].head(3)}\n  {df[offset_col].min()}')

    # Converts NA values to -inf to perform bins
    df[offset_col] = np.where(df[offset_col].isna(),-float('inf'),df[offset_col])

    bins = [-float('inf'), 0, float('inf')]
    labels = ['late', 'early']
    df[cat] = pd.cut(df[offset_col], ordered=False, bins=bins, labels=labels)
    df[cat] = np.where(df[offset_col] == -float('inf'),'not_delivered',df[cat])

    df[cat] = np.where(df[offset_col] == 0, 'on_time', df[cat])
    df[num] = np.where(df[offset_col] != -float('inf'), abs(df[offset_col]), df[offset_col])

    print(f'{df[[offset_col, cat, num]].head(3)}\n  {df[offset_col].min()}, {df[num].min()}')

    return df

df = convert_dt_offsets(df, 'carrier_offset')
df = convert_dt_offsets(df,'customer_offset')
df.head().iloc[:,5:]

df = df.drop(columns=datetime_col)

carrier_offset
0   -1.0
1   -2.0
2    2.0
Name: carrier_offset, dtype: float64
  -117.0
   carrier_offset delivered_carrier  delivered_carrier_days
0            -1.0              late                     1.0
1            -2.0              late                     2.0
2             2.0             early                     2.0
  -inf, -inf
customer_offset
0     8.0
1     2.0
2    13.0
Name: customer_offset, dtype: float64
  -189.0
   customer_offset delivered_customer  delivered_customer_days
0              8.0              early                      8.0
1              2.0              early                      2.0
2             13.0              early                     13.0
  -inf, -inf


In [50]:
df.info()
og_df_rows = len(df)

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       112650 non-null  str    
 1   order_item_id                  112650 non-null  int64  
 2   product_id                     112650 non-null  str    
 3   seller_id                      112650 non-null  str    
 4   price                          112650 non-null  float64
 5   freight_value                  112650 non-null  float64
 6   product_category_name          111047 non-null  str    
 7   product_name_lenght            111047 non-null  float64
 8   product_description_lenght     111047 non-null  float64
 9   product_photos_qty             111047 non-null  float64
 10  product_weight_g               112632 non-null  float64
 11  product_length_cm              112632 non-null  float64
 12  product_height_cm              112632 non

In [51]:
df.isna().sum()

order_id                             0
order_item_id                        0
product_id                           0
seller_id                            0
price                                0
freight_value                        0
product_category_name             1603
product_name_lenght               1603
product_description_lenght        1603
product_photos_qty                1603
product_weight_g                    18
product_length_cm                   18
product_height_cm                   18
product_width_cm                    18
product_category_name_english     1627
customer_id                          0
order_status                         0
review_id                         1491
review_score                      1491
review_comment_title             99265
review_comment_message           65477
score_class                       1491
payment_sequential                   3
payment_installments_total           3
payment_type_all                     3
seller_zip_code_prefix   

## Cleaning NA rows

In [52]:
dummy = len(df)
df1 = df[df['review_id'].isna() == False]
print(f'Remove rows with NA review_id\nPercentage of data lost: {(og_df_rows-len(df1))/og_df_rows:.2%}')

remove_rows = ['product_category_name_english',
               'product_width_cm',
               'payment_sequential'
               ]
for col in remove_rows:
    df1 = df1[df1[col].isna() == False]
    print(f'Percentage of loss from removing {col}: {df[col].isna().sum()} ({df[col].isna().sum()/len(df):.2%})')


Remove rows with NA review_id
Percentage of data lost: 1.32%
Percentage of loss from removing product_category_name_english: 1627 (1.44%)
Percentage of loss from removing product_width_cm: 18 (0.02%)
Percentage of loss from removing payment_sequential: 3 (0.00%)


In [53]:
print(f'Number of duplicated rows: {df1.duplicated().sum()}')
dupes_and_na(df1)

Number of duplicated rows: 0


,NA Values,Duplicated Values
order_id,0,13503
order_item_id,0,109526
product_id,0,77506
seller_id,0,106521
price,0,103686
freight_value,0,102601
product_category_name,0,109476
product_name_lenght,0,109481
product_description_lenght,0,106594
product_photos_qty,0,109528


### Removing identifier columns

In [54]:
df1 = df1.drop(columns=['order_id', 'order_item_id', 'product_id', 'seller_id','customer_id','customer_unique_id','seller_id','review_id'])

In [55]:
df1.info()
df1.shape

<class 'pandas.DataFrame'>
Index: 109547 entries, 0 to 112649
Data columns (total 33 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   price                          109547 non-null  float64
 1   freight_value                  109547 non-null  float64
 2   product_category_name          109547 non-null  str    
 3   product_name_lenght            109547 non-null  float64
 4   product_description_lenght     109547 non-null  float64
 5   product_photos_qty             109547 non-null  float64
 6   product_weight_g               109547 non-null  float64
 7   product_length_cm              109547 non-null  float64
 8   product_height_cm              109547 non-null  float64
 9   product_width_cm               109547 non-null  float64
 10  product_category_name_english  109547 non-null  str    
 11  order_status                   109547 non-null  str    
 12  review_score                   109547 non-null

(109547, 33)

# Prepare for EDA

In [56]:
import os
# Save consolidated dataset

output_path = "./olist_cleaned.csv"
df1.to_csv(output_path, index=False)
print(f"✅ Saved consolidated dataset to: {output_path}")
print(f"   Shape: {df1.shape[0]:,} rows × {df1.shape[1]} columns")
print(f"   Size: {os.path.getsize(output_path) / 1e6:.2f} MB")

✅ Saved consolidated dataset to: ./olist_cleaned.csv
   Shape: 109,547 rows × 33 columns
   Size: 24.53 MB
